# Token cost analysis

> Research cut-off date: 2026-07-22.
> Status: Phase 4, not yet populated. This notebook reads only from `data/`.
> It computes nothing from values typed into the notebook itself, so that no
> figure can exist here without a source row behind it.

## Scope

Computes cost per accepted task across models and workload shapes, using the formulas defined in docs/appendices/formulas.md and implemented in scripts/calculate_token_costs.py. Shows how the ranking by advertised price differs from the ranking by cost per accepted task, which is the central argument of chapter 15.

## Research checklist

- [ ] Load data/pricing.csv once it is populated in Phase 4.
- [ ] Define workload shapes from measured token counts, not from assumed ones, and record the measurement source.
- [ ] State the acceptance criterion used for every acceptance rate.
- [ ] Report cost per accepted task alongside advertised price so the divergence is visible.
- [ ] Show sensitivity to the acceptance rate, since it is usually the least well measured input.


## Load the data layer


In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parent / "data"

# The data layer is the only permitted input. A notebook that hard-codes a
# benchmark score, a price, or an energy figure is a defect: the value would
# then exist with no source_id behind it.
models = pd.read_csv(DATA / "models.csv")
sources = pd.read_csv(DATA / "sources.csv")
print(f"{len(models)} model row(s), {len(sources)} source row(s)")


## Compute costs with the reference implementation


In [ ]:
import sys

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

from calculate_token_costs import Prices, Usage, compute

pricing = pd.read_csv(DATA / "pricing.csv")

# The script is the reference implementation. If this notebook and the script
# disagree, the notebook is wrong.
example = Usage(input_fresh=12_000, input_cached=8_000, output=900, reasoning=3_400,
                acceptance_rate=0.72)
for _, row in pricing.iterrows():
    prices = Prices(float(row["input_price_per_1m"]),
                    float(row["cached_input_price_per_1m"]),
                    float(row["output_price_per_1m"]))
    result = compute(example, prices)
    print(row["model_id"], round(result.cost_per_accepted_task, 6))
